# Thrust Requirement Calculator

Info

In [95]:
# Import Dataset
import pandas as pd

# Data Source: https://www.ukssdc.ac.uk/wdcc1/msise90.html
df = pd.read_csv('fine200km_data.csv', skiprows=1)
df.rename(columns={df.columns[0]: 'Altitude'}, inplace=True)
df.rename(columns={'Total Mass (gm/cm-3)': 'Total Mass (g/cm-3)'}, inplace=True)
df.drop(['Exospheric', 'at Altitude'], axis=1, inplace=True)
print(df)

    Altitude            He             O            N2            O2  \
0         50  1.160000e+11  0.000000e+00  1.730000e+16  4.640000e+15   
1         60  3.480000e+10  0.000000e+00  5.180000e+15  1.390000e+15   
2         70  9.460000e+09  0.000000e+00  1.400000e+15  3.740000e+14   
3         80  2.050000e+09  4.780000e+09  2.960000e+14  7.840000e+13   
4         90  4.750000e+08  2.520000e+11  6.140000e+13  1.570000e+13   
5        100  1.350000e+08  5.130000e+11  1.110000e+13  2.490000e+12   
6        110  5.490000e+07  2.240000e+11  1.570000e+12  2.520000e+11   
7        120  3.080000e+07  8.880000e+10  3.360000e+11  3.570000e+10   
8        130  1.900000e+07  4.390000e+10  1.170000e+11  9.380000e+09   
9        140  1.400000e+07  2.670000e+10  5.640000e+10  3.900000e+09   
10       150  1.180000e+07  1.820000e+10  3.180000e+10  2.020000e+09   
11       160  1.050000e+07  1.330000e+10  1.960000e+10  1.180000e+09   
12       170  9.610000e+06  1.010000e+10  1.280000e+10  7.310000

In [96]:
# Helper Functions
import math

def CalculateOrbitalVelocity(OrbitAltitude: int):
    """
    @brief Function to calculate Orbtial Velocity

    v = sqrt(GM/R)
    G -> Gravitational Constant -> 6.6743 × 10^−11 m^3 kg^−1 s^−2
    M -> Mass of Earth -> 5.9722 × 10^24 kilograms
    R -> Radius of Orbit -> Radius of Earth + Orbit Altitude

    @param OrbitAltitude: Orbit altitude of satellite in meters

    @return: Orbit Veloctiy: int.
    """
    G = 6.6743 * (10**-11)
    M = 5.9722 * (10**24)
    EARTH_RADIUS = 6371000 # meters

    v = math.sqrt(((G * M)/(EARTH_RADIUS + OrbitAltitude)))
    return v


def CalculateDragForce(AtmosphericDensity: float, OrbitAltitude: int, SurfaceArea: float):
    """
    @brief Function to Calculate Drag Force

    F_drag = ½ρv²ACd
    Cd -> Drag Coefficient

    @param OrbitAltitude: Orbit altitude of satellite in meters
    @param AtmosphericDensity: Atmospheric density in kg/m^3
    @param SurfaceArea: Surface area of satellite perpendicular to air stream in m^3

    @return Drag Force in Newtons: int.
    """
    # Cd = 2.2 constant assumption (need to document where this assumption came from )
    Cd = 2.2
    oribital_velocity = CalculateOrbitalVelocity(OrbitAltitude)
    F_drag = 0.5 * AtmosphericDensity * (oribital_velocity**2) * SurfaceArea * Cd
    
    # Debug Prints
    # print(f'AtmosphericDensity : {AtmosphericDensity}')
    # print(f'oribital_velocity : {oribital_velocity}')
    # print(f'SurfaceArea : {SurfaceArea}')
    # print(f'Cd : {Cd}')
    # print(f'F_drag : {F_drag}')
    # print("\n")
    
    return F_drag

In [97]:
# Calculate Drag force to determine the Required thrust to keep the satellite in Orbit
import numpy as np

"""
A cuboid has 6 surfaces. The avergae surface area of the satellites is computed and used in the calculations

| Satellite Size | Largest S.A. (m^2) | Smallest S.A. (m^2) | Average S.A. (m^2) |
|----------------|--------------------|---------------------|--------------------|
|       3U       |        0.03        |         0.01        |       0.0233       |
|       6U       |        0.06        |         0.02        |       0.0367       |
|       12U      |        0.06        |         0.04        |       0.0533       |
"""
surface_areas = np.array([0.0233, 0.0367, 0.0533])

# Extract Column of Altitudes & Atmospheric Densities
Altitudes = df['Altitude'].to_numpy()
Densities = df['Total Mass (g/cm-3)'].to_numpy()

for idx, density in np.ndenumerate(Densities):
    # Convert densities to kg/m-3
    Densities[idx] = Densities[idx] * 1000

# Dataframe of drag forces
# Each column represents a different satellite size / surface area
# Each row represents the calculation of drag force at different altitudes
df_drag_forces = pd.DataFrame()

for surface_idx, surface_area in np.ndenumerate(surface_areas):
    temp_drag_forces_array = np.empty_like(Densities)
    temp_drag_forces_array.fill(0)

    for idx, density in np.ndenumerate(Densities):
        # Multiply by 1000 for force in mN
        temp_drag_forces_array[idx] = 1000 * CalculateDragForce(Densities[idx], Altitudes[idx], surface_areas[surface_idx])    

    df_drag_forces[f'{surface_idx}'] = temp_drag_forces_array
    

print(df_drag_forces)

            (0,)          (1,)          (2,)
0   1.715779e+06  2.702537e+06  3.924937e+06
1   5.115259e+05  8.057082e+05  1.170143e+06
2   1.379033e+05  2.172125e+05  3.154613e+05
3   2.918415e+04  4.596816e+04  6.676030e+04
4   5.997172e+03  9.446190e+03  1.371885e+04
5   1.072755e+03  1.689704e+03  2.453984e+03
6   1.489667e+02  2.346386e+02  3.407694e+02
7   3.207029e+01  5.051415e+01  7.336251e+01
8   1.144908e+01  1.803352e+01  2.619037e+01
9   5.692459e+00  8.966233e+00  1.302180e+01
10  3.319259e+00  5.228190e+00  7.592984e+00
11  2.132661e+00  3.359170e+00  4.878576e+00
12  1.452773e+00  2.288273e+00  3.323296e+00
13  1.032654e+00  1.626540e+00  2.362251e+00
14  7.552470e-01  1.189595e+00  1.727668e+00
15  5.660335e-01  8.915635e-01  1.294832e+00
16  4.313393e-01  6.794057e-01  9.867118e-01
17  3.335258e-01  5.253389e-01  7.629581e-01
18  2.613683e-01  4.116832e-01  5.978941e-01
19  2.068495e-01  3.258101e-01  4.731793e-01
20  1.651586e-01  2.601426e-01  3.778092e-01


In [98]:
# Plot the data
import plotly.express as px

df_results = pd.DataFrame()
df_results['Altitude (km)'] = Altitudes
df_results['3U'] = df_drag_forces['(0,)'].to_numpy()
df_results['6U'] = df_drag_forces['(1,)'].to_numpy()
df_results['12U'] = df_drag_forces['(2,)'].to_numpy()

# Filter out altitudes
df_filtered = df_results[df_results["Altitude (km)"] >= 150]
df_filtered = df_filtered[df_results["Altitude (km)"] < 250]

fig = px.line(
    df_filtered, 
    x="Altitude (km)", 
    y=['3U', '6U', '12U'],
    title="Force Required To Keep a Satellite in Orbit"
    )

# newcolors = ["#0B0" "#00F" "#50F" "#A0F"];

fig.update_layout(
    yaxis_title='Drag Forces (mN)'
    )

fig.show()
fig.write_html("DragForces.html")

C:\Users\shiva\AppData\Local\Temp\ipykernel_38848\3893894737.py:12: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

